# Эксперимент 01 — разведочный анализ данных
**Цель:** изучить структуру, распределения, выбросы и зависимости.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.features import add_features

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data/raw/train.csv'
taxi_data = pd.read_csv(DATA_PATH)
taxi_data.head()

In [ ]:
taxi_data.info()
taxi_data.describe(include='all').T

In [ ]:
taxi_data.isna().sum().sort_values(ascending=False)

In [ ]:
plot_data = taxi_data.loc[taxi_data['trip_duration'].between(1, 3600)]
plt.figure(figsize=(10, 6))
sns.histplot(plot_data['trip_duration'], bins=60)
plt.title('Распределение продолжительности поездок до одного часа')
plt.xlabel('Продолжительность, секунд')
plt.ylabel('Количество поездок')
plt.tight_layout()
plt.show()

In [ ]:
taxi_data['trip_duration_log'] = np.log1p(taxi_data['trip_duration'])
plt.figure(figsize=(10, 6))
sns.histplot(taxi_data['trip_duration_log'], bins=60)
plt.title('Распределение логарифмированной продолжительности поездки')
plt.xlabel('log(1 + trip_duration)')
plt.tight_layout()
plt.show()

In [ ]:
featured = add_features(taxi_data)
trips_by_hour = featured['pickup_hour'].value_counts().sort_index()
plt.figure(figsize=(10, 6))
sns.barplot(x=trips_by_hour.index, y=trips_by_hour.values)
plt.title('Распределение количества поездок по часам суток')
plt.xlabel('Час начала поездки')
plt.ylabel('Количество поездок')
plt.tight_layout()
plt.show()

In [ ]:
day_names = {0:'Понедельник',1:'Вторник',2:'Среда',3:'Четверг',4:'Пятница',5:'Суббота',6:'Воскресенье'}
trips_by_day = featured['pickup_day_of_week'].value_counts().sort_index().rename(index=day_names)
plt.figure(figsize=(11, 6))
sns.barplot(x=trips_by_day.index, y=trips_by_day.values)
plt.title('Распределение поездок по дням недели')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
pivot = pd.crosstab(featured['pickup_day_of_week'], featured['pickup_hour'])
pivot.index = [day_names[i] for i in pivot.index]
plt.figure(figsize=(15, 7))
sns.heatmap(pivot, cmap='YlOrRd')
plt.title('Распределение количества поездок по дням недели и часам суток')
plt.xlabel('Час начала поездки')
plt.ylabel('День недели')
plt.tight_layout()
plt.show()

In [ ]:
sample = featured.loc[
    featured['distance_km'].between(0.05, 30)
    & featured['trip_duration'].between(30, 7200)
].sample(n=min(50000, len(featured)), random_state=42)
plt.figure(figsize=(11, 7))
sns.scatterplot(data=sample, x='distance_km', y='trip_duration', alpha=0.2, s=15)
plt.title('Зависимость продолжительности поездки от расстояния')
plt.xlabel('Расстояние, км')
plt.ylabel('Продолжительность, секунд')
plt.tight_layout()
plt.show()

In [ ]:
numeric = featured.select_dtypes(include='number')
plt.figure(figsize=(14, 10))
sns.heatmap(numeric.corr(), annot=True, cmap='coolwarm', fmt='.2f', center=0)
plt.title('Корреляционная матрица числовых признаков')
plt.tight_layout()
plt.show()